# HIXL 数据传输

TransferSync、TransferAsync 和 GetTransferStatus 是 HIXL 数据传输的核心接口。TransferSync 进行同步传输，阻塞等待完成；TransferAsync 进行异步传输，通过 GetTransferStatus 查询状态。

理解数据传输的关键：传输是 HIXL 的最终目的，所有前面的 Initialize、RegisterMem、Connect 都是为传输做准备。

本节学习大纲如下：

- 数据传输的概念与方向语义
- TransferSync 使用方法
- TransferAsync 使用方法
- GetTransferStatus 使用方法
- 开发注意事项
- 代码示例


## 1. 数据传输的概念与方向语义

### 1.1 单边通信的概念

HIXL 的核心功能是单边通信：本端进程可以主动读写远端内存，无需远端进程参与。数据传输就是执行这种单边读写操作。

单边通信的技术特点：

- 本端发起传输请求，直接访问远端内存
- 远端进程无需感知传输过程，数据直接落入其内存
- 利用 RDMA/HCCS 硬件通道，实现高效数据搬运

### 1.2 READ/WRITE 方向语义

方向语义是开发者最容易混淆的概念，需要理解读写操作和其对应的语义。

| 操作 | 数据方向 | `local_addr` 的角色 | `remote_addr` 的角色 |
| --- | --- | --- | --- |
| `READ` | 从远端读到本地 | 本地接收缓冲区 | 远端源数据地址 |
| `WRITE` | 从本地写到远端 | 本地源数据地址 | 远端接收缓冲区 |

- **READ**：把远端数据"读回来"，本地是目的地

    <img src="./images/read_direction.png" alt="READ方向语义图" width="60%">

- **WRITE**：把本地数据"写过去"，远端是目的地

    <img src="./images/write_direction.png" alt="WRITE方向语义图" width="60%">


### 1.3 local_addr 与 remote_addr 的区别

- **local_addr**：本地内存地址，来自本地 `aclrtMalloc` + `RegisterMem`
- **remote_addr**：远端内存地址数值，由远端分配注册后通过控制面传输过来

**常见错误**：把本地的 `MemHandle` 当作 `remote_addr` 填入。

### 1.4 同步与异步的区别

| 模式 | 接口 | 完成方式 | 适用场景 |
| --- | --- | --- | --- |
| 同步 | `TransferSync` | 阻塞等待完成 | 简单传输、调试验证 |
| 异步 | `TransferAsync` + `GetTransferStatus` | 提交后轮询状态 | 高性能场景、重叠计算与传输 |

## 2. TransferSync 使用方法

### 2.1 接口签名

```cpp
Status TransferSync(const AscendString &remote_engine,
                    TransferOp operation,
                    const std::vector<TransferOpDesc> &op_descs,
                    int32_t timeout_in_millis = 1000);
```

### 2.2 参数说明

`TransferOpDesc` 描述一次传输操作的地址和长度：

```cpp
struct TransferOpDesc {
  uintptr_t local_addr;   // 本地地址
  uintptr_t remote_addr;  // 远端地址数值
  size_t len;             // 传输长度
};
```

关键参数理解：

- **remote_engine**：与 Connect 时使用的相同，标识传输目标
- **operation**：`READ` 或 `WRITE`，决定数据方向
- **op_descs**：可以包含多个描述符，一次下发多个传输请求
- **timeout_in_millis**：传输超时时间，阻塞等待的最长时间


### 2.3 返回值处理

| 返回值 | 含义 | 处理建议 |
| --- | --- | --- |
| `SUCCESS` | 传输成功 | 继续后续操作或断链清理 |
| `PARAM_INVALID` | 参数错误 | 检查地址是否已注册、长度是否合法 |
| `NOT_CONNECTED` | 未建链 | 先完成 Connect |
| `TIMEOUT` | 传输超时 | 检查远端状态、网络是否通畅 |
| `RESOURCE_EXHAUSTED` | 资源耗尽 | 减少并发请求或等待资源释放 |
| 其他 | 失败 | 打印返回码和 `aclGetRecentErrMsg()` |



## 3. TransferAsync 使用方法

### 3.1 接口签名

```cpp
Status TransferAsync(const AscendString &remote_engine,
                     TransferOp operation,
                     const std::vector<TransferOpDesc> &op_descs,
                     const TransferArgs &optional_args,
                     TransferReq &req);
```

### 3.2 参数说明

与 TransferSync 的区别在于：

- **optional_args**：可选参数（预留），保持默认初始化即可
- **req**：输出请求句柄，用于后续 `GetTransferStatus` 查询

**注意**：`TransferAsync` 返回 `SUCCESS` 只表示请求下发成功，不表示传输完成，必须根据 `GetTransferStatus` 查询到的状态来进行判断。

### 3.3 返回值处理

| 返回值 | 含义 | 处理建议 |
| --- | --- | --- |
| `SUCCESS` | 请求下发成功 | 用 `GetTransferStatus` 查询完成状态 |
| `NOT_CONNECTED` | 未建链 | 先完成 Connect |
| `RESOURCE_EXHAUSTED` | 资源耗尽 | 减少并发请求或等待资源释放 |
| 其他 | 失败 | 打印返回码和 `aclGetRecentErrMsg()` |

### 3.4 约束说明

- 与 Initialize 同线程调用，或通过 context 切换


## 4. GetTransferStatus 使用方法

### 4.1 接口签名

```cpp
Status GetTransferStatus(const TransferReq &req, TransferStatus &status);
```

### 4.2 参数说明

- **req**：`TransferAsync` 输出的请求句柄
- **status**：输出传输状态

`TransferStatus` 的可能值：

| 状态 | 含义 | 处理建议 |
| --- | --- | --- |
| `WAITING` | 传输进行中 | 继续轮询等待 |
| `COMPLETED` | 传输成功完成 | 停止查询，继续后续操作 |
| `FAILED` | 传输失败 | 停止查询，进入错误处理 |
| `TIMEOUT` | 暂不支持 | 异步传输超时需用户自行判断 |

### 4.3 返回值处理

| 返回值 | 含义 |
| --- | --- |
| `SUCCESS` | 查询成功，status 已更新 |
| `PARAM_INVALID` | req 无效或已查询过终态 |
| `NOT_CONNECTED` | 链路已断开 |
| 其他 | 失败 |

### 4.4 轮询流程

查询到 `COMPLETED` 或 `FAILED` 后，相关资源会释放，不支持再次查询同一个 `req`。

推荐的轮询流程：

```
TransferAsync → 得到 req
  → 周期性 GetTransferStatus
  → WAITING: 继续等待
  → COMPLETED: 传输完成，停止查询
  → FAILED: 传输失败，停止查询并进入错误处理
```

异步传输超时需用户自行判断。若超时建议 Disconnect 销毁链路，避免资源泄漏。

## 5. 开发注意事项

### 5.1 READ/WRITE 方向混淆

错误示例：想把本地数据发给远端，却用了 `READ` 导致方向反了，应该用 `WRITE`，把本地数据"写过去"

### 5.2 异步传输误判完成

`TransferAsync` 返回 `SUCCESS` 只表示请求下发成功，不表示传输完成。

错误做法：TransferAsync 成功后立即 DeregisterMem → 内存还在被传输，解注册失败

正确做法：用 `GetTransferStatus` 查询到 `COMPLETED` 后才进行后续清理。

### 5.3 异步超时处理

HIXL 的 `TransferStatus::TIMEOUT` 暂不支持，异步传输超时需用户自行判断。

建议做法：

- 设置自己的超时计时器
- 若超时，调用 Disconnect 销毁链路，避免资源泄漏
- 排查建链状态、地址注册、远端生命周期和网络状态

### 5.4 多线程 context 管理

TransferSync/TransferAsync/GetTransferStatus 要求在 Initialize 同线程调用，或通过：

- `aclrtGetCurrentContext` 获取当前 context
- `aclrtSetCurrentContext` 切换到 Initialize 所在 context

跨线程调用会导致 context 不匹配，接口返回错误。


## 6. 代码示例

本节介绍了 HIXL 数据传输的核心接口 TransferSync、TransferAsync 和 GetTransferStatus。至此，HIXL 的所有核心 API 均已介绍完毕。下面的代码将给出一个完整的调用流程示例，展示这些 API 如何协同工作。

```c++
constexpr int32_t kDeviceId = 0;
constexpr size_t kBytes = 4 * 1024 * 1024;
const AscendString kLocalEngine = "10.10.10.1:26000";
const AscendString kRemoteEngine = "10.10.10.2:26000";

aclrtSetDevice(kDeviceId);

Hixl engine;
std::map<AscendString, AscendString> options;
options[OPTION_BUFFER_POOL] = "0:0";
options[OPTION_AUTO_CONNECT] = "0";

Status ret = engine.Initialize(kLocalEngine, options);
if (ret != SUCCESS) {
    printf("[ERROR] Initialize failed, ret = %u\n", ret);
    return -1;
}

void *device_buffer = nullptr;
MemHandle handle = nullptr;
aclrtMalloc(&device_buffer, kBytes, ACL_MEM_MALLOC_HUGE_ONLY);

MemDesc desc{};
desc.addr = reinterpret_cast<uintptr_t>(device_buffer);
desc.len = kBytes;
engine.RegisterMem(desc, MEM_DEVICE, handle);

ret = engine.Connect(kRemoteEngine, 3000);
if (ret != SUCCESS && ret != ALREADY_CONNECTED) {
    printf("[ERROR] Connect failed, ret = %u\n", ret);
    engine.DeregisterMem(handle);
    aclrtFree(device_buffer);
    engine.Finalize();
    aclrtResetDevice(kDeviceId);
    return -1;
}

// 同步传输示例
TransferOpDesc sync_desc{};
sync_desc.local_addr = reinterpret_cast<uintptr_t>(device_buffer);
sync_desc.remote_addr = 0;  // 实际应由远端交换得到
sync_desc.len = kBytes;

std::vector<TransferOpDesc> sync_descs{sync_desc};
ret = engine.TransferSync(kRemoteEngine, WRITE, sync_descs, 3000);
if (ret != SUCCESS) {
    printf("[ERROR] TransferSync failed, ret = %u\n", ret);
}

// 异步传输示例
TransferReq req;
TransferArgs optional_args{};
ret = engine.TransferAsync(kRemoteEngine, WRITE, sync_descs, optional_args, req);
if (ret != SUCCESS) {
    printf("[ERROR] TransferAsync failed, ret = %u\n", ret);
} else {
    // 轮询查询状态
    TransferStatus status = TransferStatus::WAITING;
    while (status == TransferStatus::WAITING) {
        ret = engine.GetTransferStatus(req, status);
        if (ret != SUCCESS) break;
        std::this_thread::sleep_for(std::chrono::milliseconds(1));
    }
    if (status == TransferStatus::COMPLETED) {
        printf("TransferAsync completed\n");
    } else {
        printf("[ERROR] TransferAsync failed, status = %u\n", static_cast<uint32_t>(status));
    }
}

// 断链和清理
engine.Disconnect(kRemoteEngine, 3000);
engine.DeregisterMem(handle);
aclrtFree(device_buffer);
engine.Finalize();
aclrtResetDevice(kDeviceId);
```

## 课后练习

本节介绍了数据传输的概念、TransferSync/TransferAsync/GetTransferStatus 的使用方法和开发注意事项，请根据学习内容完成以下题目进行自测。

1. （判断题）`TransferSync` 返回 `SUCCESS` 表示远端业务逻辑已经消费了数据。

2. （判断题）`TransferAsync` 返回 `SUCCESS` 表示传输已完成。

3. （单选题）`TransferOp::READ` 的方向语义是？
   A. 从本地读到远端，`remote_addr` 是远端接收地址
   B. 从远端读到本地，`local_addr` 是本地接收地址
   C. 从本地写到远端，`local_addr` 是本地源地址
   D. 只在 Host 内存之间传输

4. （单选题）`remote_addr`（远端地址）应如何获取？
   A. 本地随意构造
   B. 远端 `RegisterMem` 输出的 `MemHandle`
   C. 远端分配并注册的地址数值，通过控制面交换得到
   D. 本地 `RegisterMem` 输出的 `MemHandle`

5. （多选题）关于 `TransferAsync` 和 `GetTransferStatus`，哪些说法正确？
   A. `TransferAsync` 成功只表示请求下发成功
   B. 最终完成态需要通过 `GetTransferStatus` 查询
   C. 查询到 `COMPLETED` 或 `FAILED` 后，不应继续查询同一个请求
   D. 异步超时完全由 HIXL 自动通过 `TransferStatus::TIMEOUT` 上报

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/02.05_answer.txt